In [0]:
spark.conf.set(
    "fs.azure.account.key.tokyoolympadls.blob.core.windows.net",
    "Your_Access_key"
)

In [0]:
display(dbutils.fs.ls("wasbs://tokyo-olympics-data@tokyoolympadls.blob.core.windows.net/raw-data/"))


path,name,size,modificationTime
wasbs://tokyo-olympics-data@tokyoolympadls.blob.core.windows.net/raw-data/EntriesGender.csv,EntriesGender.csv,1076,1760651637000
wasbs://tokyo-olympics-data@tokyoolympadls.blob.core.windows.net/raw-data/athletes.csv,athletes.csv,407406,1760651605000
wasbs://tokyo-olympics-data@tokyoolympadls.blob.core.windows.net/raw-data/coaches.csv,coaches.csv,16494,1760651623000
wasbs://tokyo-olympics-data@tokyoolympadls.blob.core.windows.net/raw-data/medals.csv,medals.csv,2320,1760651652000
wasbs://tokyo-olympics-data@tokyoolympadls.blob.core.windows.net/raw-data/teams.csv,teams.csv,34526,1760651665000


In [0]:
spark

In [0]:
athletes = spark.read.csv(
    "wasbs://tokyo-olympics-data@tokyoolympadls.blob.core.windows.net/raw-data/athletes.csv",
    header=True,
    inferSchema=True
)
EntriesGender = spark.read.csv(
    "wasbs://tokyo-olympics-data@tokyoolympadls.blob.core.windows.net/raw-data/EntriesGender.csv",
    header=True,
    inferSchema=True
)
coaches = spark.read.csv(
    "wasbs://tokyo-olympics-data@tokyoolympadls.blob.core.windows.net/raw-data/coaches.csv",
    header=True,
    inferSchema=True
)
medals = spark.read.csv(
    "wasbs://tokyo-olympics-data@tokyoolympadls.blob.core.windows.net/raw-data/medals.csv",
    header=True,
    inferSchema=True
)
teams = spark.read.csv(
    "wasbs://tokyo-olympics-data@tokyoolympadls.blob.core.windows.net/raw-data/teams.csv",
    header=True,
    inferSchema=True
)

In [0]:
athletes.printSchema()
EntriesGender.printSchema(10)
coaches.printSchema()
medals.printSchema()
teams.printSchema()


root
 |-- PersonName: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Discipline: string (nullable = true)

root
 |-- Discipline: string (nullable = true)
 |-- Female: integer (nullable = true)
 |-- Male: integer (nullable = true)
 |-- Total: integer (nullable = true)

root
 |-- Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Discipline: string (nullable = true)
 |-- Event: string (nullable = true)

root
 |-- Rank: integer (nullable = true)
 |-- TeamCountry: string (nullable = true)
 |-- Gold: integer (nullable = true)
 |-- Silver: integer (nullable = true)
 |-- Bronze: integer (nullable = true)
 |-- Total: integer (nullable = true)
 |-- Rank by Total: integer (nullable = true)

root
 |-- TeamName: string (nullable = true)
 |-- Discipline: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Event: string (nullable = true)

<class 'list'>
['PersonName', 'Country', 'Discipline']


In [0]:
from pyspark.sql.functions import col, isnull, initcap, lower, upper, col,trim ,monotonically_increasing_id
from functools import reduce

def Dropduplicates(df):
    print("Nombre de lignes avant suppression :", df.count())
    df_clean = df.dropDuplicates(df.columns)
    print("Nombre de lignes après suppression :", df_clean.count())
    return df_clean
def NullCol(df):
    condition = reduce(lambda x, y: x | y, [col(c).isNull() for c in df.columns])
    null_rows = df.filter(condition).count()
    print(f"Nombre de lignes contenant au moins une valeur nulle : {null_rows}")

def CleanText(df, cols):  
    """Supprimer les espaces et uniformiser la casse"""
    string_cols = [c for c in cols if dict(df.dtypes)[c] == "string"]
    
    for c in string_cols:
        df = df.withColumn(c, initcap(trim(col(c))))
    
    print(f"Colonnes texte nettoyées : {string_cols}")
    return df   
    


In [0]:
athletes_clean = Dropduplicates(athletes)
NullCol(athletes_clean)
athletes_clean = CleanText(athletes_clean, athletes_clean.columns)
athletes_normalized = (
    athletes_clean
    .withColumnRenamed("PersonName", "athlete_name")
    .withColumnRenamed("Country", "country")
    .withColumnRenamed("Discipline", "sport")
)
athletes_enriched = athletes_normalized.withColumn(
    "athlete_id", monotonically_increasing_id()
)
cols = ["athlete_id"] + [col for col in athletes_enriched.columns if col != "athlete_id"]
athletes_enriched = athletes_enriched.select(cols)
athletes_enriched.printSchema()

Nombre de lignes avant suppression : 11085
Nombre de lignes après suppression : 11084
Nombre de lignes contenant au moins une valeur nulle : 0
Colonnes texte nettoyées : ['PersonName', 'Country', 'Discipline']
root
 |-- athlete_id: long (nullable = false)
 |-- athlete_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- sport: string (nullable = true)



In [0]:
EntriesGender_clean = Dropduplicates(EntriesGender)
NullCol(EntriesGender_clean)
EntriesGender_clean = CleanText(EntriesGender_clean, EntriesGender_clean.columns)

EntriesGender_normalized = (
    EntriesGender_clean
    .withColumnRenamed("Discipline", "sport"))
EntriesGender_normalized.printSchema()

Nombre de lignes avant suppression : 46
Nombre de lignes après suppression : 46
Nombre de lignes contenant au moins une valeur nulle : 0
Colonnes texte nettoyées : ['Discipline']
root
 |-- sport: string (nullable = true)
 |-- Female: integer (nullable = true)
 |-- Male: integer (nullable = true)
 |-- Total: integer (nullable = true)



In [0]:
coaches_clean = Dropduplicates(coaches)
NullCol(coaches_clean)
coaches_clean = coaches_clean.drop("Event")
NullCol(coaches_clean)
coaches_clean = CleanText(coaches_clean, coaches_clean.columns)
coaches_normalized = (
    coaches_clean
    .withColumnRenamed("Name", "coach_name")
    .withColumnRenamed("Country", "country")
    .withColumnRenamed("Discipline", "sport"))
coaches_enriched = coaches_normalized.withColumn("coach_id", monotonically_increasing_id())
cols = ["coach_id"]+[col for col in coaches_enriched.columns if col !="coach_id"]
coaches_enriched = coaches_enriched.select(cols)
coaches_enriched.printSchema()

Nombre de lignes avant suppression : 394
Nombre de lignes après suppression : 393
Nombre de lignes contenant au moins une valeur nulle : 145
Nombre de lignes contenant au moins une valeur nulle : 0
Colonnes texte nettoyées : ['Name', 'Country', 'Discipline']
root
 |-- coach_id: long (nullable = false)
 |-- coach_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- sport: string (nullable = true)



In [0]:
medals_clean = Dropduplicates(medals)
NullCol(medals_clean)
medals_clean = CleanText(medals_clean, medals_clean.columns)
medals_normalized = (
    medals_clean
    .withColumnRenamed("TeamCountry", "country")
    .withColumnRenamed("Rank by Total","rank_total"))
medals_normalized.printSchema()

Nombre de lignes avant suppression : 93
Nombre de lignes après suppression : 93
Nombre de lignes contenant au moins une valeur nulle : 0
Colonnes texte nettoyées : ['TeamCountry']
root
 |-- Rank: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- Gold: integer (nullable = true)
 |-- Silver: integer (nullable = true)
 |-- Bronze: integer (nullable = true)
 |-- Total: integer (nullable = true)
 |-- rank_total: integer (nullable = true)



In [0]:
teams_clean = Dropduplicates(teams)
NullCol(teams_clean)
teams_clean = CleanText(teams_clean, teams_clean.columns)
teams_normalized = (
    teams_clean
    .withColumnRenamed("TeamName", "team_name")
    .withColumnRenamed("Country", "country")
    .withColumnRenamed("Discipline", "sport")
    .withColumnRenamed("Event", "event"))
teams_enriched = teams_normalized.withColumn("team_id", monotonically_increasing_id())
cols =["team_id"] + [col for col in teams_enriched.columns if col !="team_id" ]
teams_enriched = teams_enriched.select(cols)
teams_enriched.printSchema()


Nombre de lignes avant suppression : 743
Nombre de lignes après suppression : 743
Nombre de lignes contenant au moins une valeur nulle : 0
Colonnes texte nettoyées : ['TeamName', 'Discipline', 'Country', 'Event']
root
 |-- team_id: long (nullable = false)
 |-- team_name: string (nullable = true)
 |-- sport: string (nullable = true)
 |-- country: string (nullable = true)
 |-- event: string (nullable = true)

+-------+-------------+-------------------+--------------------+------------+
|team_id|    team_name|              sport|             country|       event|
+-------+-------------+-------------------+--------------------+------------+
|      0|Great Britain|Artistic Gymnastics|       Great Britain|Women's Team|
|      1|      Nigeria|         Basketball|             Nigeria|       Women|
|      2|  Netherlands|           Handball|         Netherlands|       Women|
|      3|        Japan|         Volleyball|               Japan|         Men|
|      4|United States|            Archery|

In [0]:
joined_Ath_Coach_df = athletes_enriched.join(
    coaches_enriched,
    on="sport",      # colonne commune
    how="inner"      # type de jointure (inner, left, right, etc.)
)
joined_Ath_Coach_df.show(5)

+-----------------+----------+----------------+-------------+--------+-----------------+--------------------+
|            sport|athlete_id|    athlete_name|      country|coach_id|       coach_name|             country|
+-----------------+----------+----------------+-------------+--------+-----------------+--------------------+
|Baseball/softball|        10|Gasparotto Marta|        Italy|     384|  Weinstein Jerry|United States Of ...|
|         Football|        11|       Gil Bryan|        Spain|     392|       Riise Hege|       Great Britain|
|         Football|        15|  Kennedy Alanna|    Australia|     392|       Riise Hege|       Great Britain|
|Artistic Swimming|        18|  Kulagina Daria|      Belarus|     379|Farinelli Roberta|               Italy|
|         Football|        26|      Little Kim|Great Britain|     392|       Riise Hege|       Great Britain|
+-----------------+----------+----------------+-------------+--------+-----------------+--------------------+
only showi